# 04 — Ablation: what does each feature block contribute?

Removes the **stability** block, then the **ligand** block, then the
**structural** block, and prices each one. Then asks the question that actually
decides whether the model is useful: **did it learn structure–function signal,
or did it learn which compounds are promiscuous?**

## Two kinds of ablation, both reported

1. **Retrain-without** — drop the block, refit from scratch, measure the loss.
   This is the honest cost of not *having* the data, which is what matters when
   the block is expensive to obtain (structural features need a structure and a
   pocket definition).
2. **Permutation** — keep the trained model, shuffle the block's columns. This
   measures how much the *fitted* model leans on the block. Blocks are permuted
   jointly, never column-by-column: 256 fingerprint bits over 12 compounds are
   massively redundant, so single-column permutation would report that no bit
   matters — true, and completely uninformative about whether chemistry matters.

The two can disagree, and the disagreement is informative: a block that is
cheap to permute but costly to remove was substitutable by other features.


> ## ⚠ These results are computed on SYNTHETIC data
>
> The curated benchmark this package is designed to consume
> ([`amenabench`](https://github.com/xX-its-amit-Xx/amenabench)) currently
> contains only a `LICENSE` file — there is no data in it. Every number and
> figure below is therefore produced by
> `foldrescue.data.SyntheticCohortAdapter`, a seeded generator whose labels
> encode a mixture **we chose**.
>
> **What this notebook demonstrates:** that the pipeline runs end to end, and
> that the diagnostics behave as designed against a known ground truth.
>
> **What it does not demonstrate:** anything about pharmacological chaperone
> biology. These are not findings. Do not cite these numbers.


In [1]:
import sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")          # cookbook/ -> repo root
sys.path.insert(0, ".")
from _viz import (SERIES, TIER_COLOR, BLOCK_COLOR, INK, INK_SECONDARY,
                  INK_MUTED, use_style, save, label_bars)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
use_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("numpy", np.__version__, "| pandas", pd.__version__)


numpy 2.5.3 | pandas 3.0.5


In [2]:
from foldrescue.data import load_cohort
from foldrescue.features import FeatureSpec, build_features
from foldrescue.models import build_model
from foldrescue.evaluate import evaluate_predictions, aggregate_over_seeds
from foldrescue.attribution import (permutation_importance_blocks, interrogate_promiscuity,
                                    shap_values_for_tree_model, shap_block_shares,
                                    permutation_importance_features)

ECFP, SEEDS = 256, [0, 1, 2, 3, 4]
SPLITS = ["random", "gene_holdout"]
FULL = ("stability", "structural", "ligand", "interaction")

# "interaction" rides with its partners: it is products of protein x ligand
# terms, so removing the ligand block necessarily removes it too.
CONFIGS = {
    "full":              FULL,
    "minus_stability":   ("structural", "ligand", "interaction"),
    "minus_ligand":      ("stability", "structural"),
    "minus_structural":  ("stability", "ligand", "interaction"),
    "minus_interaction": ("stability", "structural", "ligand"),
}
for k, v in CONFIGS.items():
    print(f"{k:19s} {v}")

full                ('stability', 'structural', 'ligand', 'interaction')
minus_stability     ('structural', 'ligand', 'interaction')
minus_ligand        ('stability', 'structural')
minus_structural    ('stability', 'ligand', 'interaction')
minus_interaction   ('stability', 'structural', 'ligand')


In [3]:
rows = []
for seed in SEEDS:
    cohort = load_cohort("synthetic", seed=seed)
    for name, blocks in CONFIGS.items():
        fm = build_features(cohort, spec=FeatureSpec(blocks=blocks, ecfp_bits=ECFP))
        for split in SPLITS:
            s = cohort.split(split)
            model = build_model("tier3_structure", random_state=seed).fit(
                fm.X[s["train"]], fm.y[s["train"]],
                eval_set=(fm.X[s["calib"]], fm.y[s["calib"]]),
                feature_names=fm.feature_names)
            m = evaluate_predictions(
                fm.y[s["test"]], model.predict_proba(fm.X[s["test"]]),
                split=split, tier=name,
                y_calib=fm.y[s["calib"]], p_calib=model.predict_proba(fm.X[s["calib"]]),
                oracle_p_test=cohort.oracle_p[s["test"]])
            r = m.as_row(); r["seed"] = seed; r["n_features"] = fm.X.shape[1]
            rows.append(r)

abl = pd.DataFrame(rows)
agg = aggregate_over_seeds(abl.to_dict("records"), by=("split", "tier"))
print(agg[["split","tier","pr_auc_mean","pr_auc_std","mcc_mean","n_features_mean"]].round(4).to_string(index=False))

2026-10-10 11:53:41,572 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1308 pairs, prevalence 0.312


2026-10-10 11:53:41,596 INFO    foldrescue.features: built features: (1308, 300) (float64)


2026-10-10 11:53:43,207 INFO    foldrescue.features: built features: (1308, 283) (float64)


2026-10-10 11:53:44,136 INFO    foldrescue.features: built features: (1308, 25) (float64)


2026-10-10 11:53:44,366 INFO    foldrescue.features: built features: (1308, 289) (float64)


...

2026-10-10 11:53:57,874 INFO    foldrescue.features: built features: (1309, 283) (float64)


2026-10-10 11:53:58,668 INFO    foldrescue.features: built features: (1309, 25) (float64)


2026-10-10 11:53:59,086 INFO    foldrescue.features: built features: (1309, 289) (float64)


2026-10-10 11:53:59,769 INFO    foldrescue.features: built features: (1309, 293) (float64)


       split              tier  pr_auc_mean  pr_auc_std  mcc_mean  n_features_mean
gene_holdout              full       0.4265      0.0475    0.1785            300.0
gene_holdout minus_interaction       0.3979      0.0278    0.0924            293.0
gene_holdout      minus_ligand       0.3005      0.0315   -0.0379             25.0
gene_holdout   minus_stability       0.4122      0.0546    0.0759            283.0
gene_holdout  minus_structural       0.4291      0.0509    0.1639            289.0
      random              full       0.4365      0.0818    0.1358            300.0
      random minus_interaction       0.4001      0.0703    0.1135            293.0
      random      minus_ligand       0.3162      0.0176    0.0177             25.0
      random   minus_stability       0.4141      0.1020    0.1219            283.0
      random  minus_structural       0.4183      0.0847    0.1832            289.0


In [4]:
delta = []
for split in SPLITS:
    sub = agg[agg["split"] == split].set_index("tier")
    base = sub.loc["full", "pr_auc_mean"]
    base_sd = sub.loc["full", "pr_auc_std"]
    for name in CONFIGS:
        if name == "full":
            continue
        d = sub.loc[name, "pr_auc_mean"] - base
        pooled = float(np.hypot(base_sd, sub.loc[name, "pr_auc_std"]))
        delta.append({"split": split, "removed": name.replace("minus_", ""),
                      "pr_auc": sub.loc[name, "pr_auc_mean"],
                      "delta_vs_full": d, "pooled_sd": pooled,
                      "beyond_noise": bool(abs(d) > pooled)})
delta = pd.DataFrame(delta).round(4)
print(delta.to_string(index=False))
delta.to_csv("../results/DEMO_synthetic_ablation.csv", index=False)

       split     removed  pr_auc  delta_vs_full  pooled_sd  beyond_noise
      random   stability  0.4141        -0.0224     0.1307         False
      random      ligand  0.3162        -0.1203     0.0836          True
      random  structural  0.4183        -0.0182     0.1178         False
      random interaction  0.4001        -0.0364     0.1078         False
gene_holdout   stability  0.4122        -0.0143     0.0723         False
gene_holdout      ligand  0.3005        -0.1260     0.0570          True
gene_holdout  structural  0.4291         0.0026     0.0696         False
gene_holdout interaction  0.3979        -0.0286     0.0550         False


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.3), sharey=True)
blocks = ["stability", "ligand", "structural", "interaction"]

for ax, split in zip(axes, SPLITS, strict=True):
    sub = delta[delta["split"] == split].set_index("removed")
    vals = [sub.loc[b, "delta_vs_full"] for b in blocks]
    errs = [sub.loc[b, "pooled_sd"] for b in blocks]
    bars = ax.bar(range(len(blocks)), vals, yerr=errs, capsize=3.5, width=0.62,
                  color=[BLOCK_COLOR[b] for b in blocks], zorder=3)
    for b, v, e in zip(bars, vals, errs, strict=True):
        ax.annotate(f"{v:+.3f}", (b.get_x()+b.get_width()/2, v - e - 0.012 if v < 0 else v + e + 0.004),
                    ha="center", va="top" if v < 0 else "bottom", fontsize=7.6, color=INK_SECONDARY)
    ax.axhline(0, color=INK, lw=1.2, zorder=2)
    ax.set_xticks(range(len(blocks)))
    ax.set_xticklabels(blocks, rotation=15, ha="right")
    ax.set_title(split.replace("_", " "))

axes[0].set_ylabel("change in PR-AUC when the block is REMOVED\n(more negative = more valuable)")
fig.suptitle("Retrain-without ablation (mean +/- pooled SD over 5 seeds)",
             y=1.04, fontsize=12, color=INK)
save(fig, "04_ablation_retrain.png")
plt.show()

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


wrote results/figures/04_ablation_retrain.png


### Permutation importance on the full model

In [6]:
cohort = load_cohort("synthetic", seed=0)
fm = build_features(cohort, spec=FeatureSpec(blocks=FULL, ecfp_bits=ECFP))
s = cohort.split("random")
model = build_model("tier3_structure", random_state=0).fit(
    fm.X[s["train"]], fm.y[s["train"]],
    eval_set=(fm.X[s["calib"]], fm.y[s["calib"]]), feature_names=fm.feature_names)

imp = permutation_importance_blocks(model, fm, rows=s["test"], n_repeats=5)
print(imp.summary(), "\n")
print(imp.to_frame().to_string(index=False))

sv = shap_values_for_tree_model(model, fm, rows=s["test"])
shares = shap_block_shares(sv, fm)
print("\nSHAP absolute-mass share by block (mass, not count -- 256 fingerprint")
print("columns will always outnumber 17 stability columns):")
for k, v in sorted(shares.items(), key=lambda kv: -kv[1]):
    print(f"  {k:13s} {v:.4f}")

2026-10-10 11:54:01,137 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1308 pairs, prevalence 0.312


2026-10-10 11:54:01,160 INFO    foldrescue.features: built features: (1308, 300) (float64)


2026-10-10 11:54:01,291 INFO    foldrescue.attribution: block permutation importance: baseline pr_auc=0.5445 | block drops: interaction: +0.1647, ligand: +0.1388, stability: +0.0698, structural: +0.0217


baseline pr_auc=0.5445 | block drops: interaction: +0.1647, ligand: +0.1388, stability: +0.0698, structural: +0.0217 

      block  pr_auc_drop  pr_auc_drop_sd
interaction     0.164724        0.038256
     ligand     0.138837        0.020144
  stability     0.069797        0.021203
 structural     0.021715        0.012143

SHAP absolute-mass share by block (mass, not count -- 256 fingerprint
columns will always outnumber 17 stability columns):
  interaction   0.3327
  stability     0.2877
  ligand        0.2831
  structural    0.0965


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.3))
blocks_present = [b for b in blocks if b in imp.drops]

ax = axes[0]
vals = [imp.drops[b] for b in blocks_present]
errs = [imp.sds[b] for b in blocks_present]
bars = ax.bar(range(len(blocks_present)), vals, yerr=errs, capsize=3.5, width=0.62,
              color=[BLOCK_COLOR[b] for b in blocks_present], zorder=3)
label_bars(ax, bars, vals, dy=max(errs)+0.004)
ax.set_xticks(range(len(blocks_present)))
ax.set_xticklabels(blocks_present, rotation=15, ha="right")
ax.set_ylabel("PR-AUC drop when block is permuted")
ax.set_title("Permutation importance (block-wise)")

ax = axes[1]
vals2 = [shares.get(b, 0.0) for b in blocks_present]
bars2 = ax.bar(range(len(blocks_present)), vals2, width=0.62,
               color=[BLOCK_COLOR[b] for b in blocks_present], zorder=3)
label_bars(ax, bars2, vals2, dy=0.006)
ax.set_xticks(range(len(blocks_present)))
ax.set_xticklabels(blocks_present, rotation=15, ha="right")
ax.set_ylabel("share of total |SHAP| mass")
ax.set_title("SHAP mass share")

fig.suptitle("How the fitted model leans on each block", y=1.04, fontsize=12, color=INK)
save(fig, "04_permutation_and_shap.png")
plt.show()

wrote results/figures/04_permutation_and_shap.png


### Most informative individual features

Fingerprint bits are deliberately included in the screen, so the ranking is not
rigged in favour of the interpretable columns.

In [8]:
top = permutation_importance_features(model, fm, rows=s["test"], n_repeats=3, top_k=15)
print(top.round(4).to_string(index=False))

                                            feature       block  pr_auc_drop  pr_auc_drop_sd
                   inter_dist_to_pocket_x_desc_TPSA interaction       0.0471          0.0031
                        inter_ddg_pred_x_desc_MolWt interaction       0.0459          0.0099
                      inter_ddg_pred_x_desc_MolLogP interaction       0.0383          0.0198
                   inter_dist_decay_8A_x_desc_MolWt interaction       0.0297          0.0075
                      inter_rsa_x_desc_FractionCSP3 interaction       0.0232          0.0027
                                      residue_depth   stability       0.0223          0.0130
                                         desc_MolWt      ligand       0.0217          0.0110
                                           ddg_pred   stability       0.0205          0.0104
inter_pocket_vol_delta_abs_x_desc_NumRotatableBonds interaction       0.0199          0.0021
                                       conservation   stability       

## The question that decides whether any of this is useful

A model can score well without learning any structure–function relationship,
simply by learning **which compounds rescue a lot of things**. If a third of
your positives come from three promiscuous compounds, "predict the compound's
base rate" is a strong baseline that looks like insight.

Neither SHAP nor permutation importance detects this. Both explain *the model*,
and a model leaning hard on ligand features is entirely consistent with pure
promiscuity recall.

The interrogation below strips away each way to cheat:

1. **compound base-rate baseline** — ignore the variant, predict each compound's
   training positive rate. Any claim of pair-level modelling must beat this.
2. **variant base-rate baseline** — the symmetric check.
3. **within-variant compound shuffle** — permute compound identity among pairs
   sharing a variant. Compound marginals are preserved *exactly*, so a
   promiscuity-only model is untouched; genuine pair-specific signal dies.
4. **recovered vs. generative pair-signal share** — the synthetic cohort knows
   its own mixture weights, so the diagnostic itself can be validated. Real data
   never offers this check, which is exactly why it is worth running here.


In [9]:
recs = []
for seed in SEEDS:
    c = load_cohort("synthetic", seed=seed)
    f = build_features(c, spec=FeatureSpec(blocks=FULL, ecfp_bits=ECFP))
    sp = c.split("random")
    mdl = build_model("tier3_structure", random_state=seed).fit(
        f.X[sp["train"]], f.y[sp["train"]],
        eval_set=(f.X[sp["calib"]], f.y[sp["calib"]]), feature_names=f.feature_names)
    f_test = build_features(c, spec=FeatureSpec(blocks=FULL, ecfp_bits=ECFP), rows=sp["test"])
    sv_i = shap_values_for_tree_model(mdl, f_test)
    rep = interrogate_promiscuity(
        mdl, f_test,
        train_pairs=c.pairs.iloc[sp["train"]],
        test_pairs=c.pairs.iloc[sp["test"]].reset_index(drop=True),
        cohort=c, n_shuffles=5,
        shap_block_share=shap_block_shares(sv_i, f_test))
    r = rep.to_frame().iloc[0].to_dict(); r["seed"] = seed
    recs.append(r)
    if seed == 0:
        print("seed 0 verdict:\n ", rep.verdict(), "\n")

prom = pd.DataFrame(recs)
cols = ["seed","model","compound_baseline","variant_baseline","within_variant_shuffled",
        "prevalence","pair_specificity","ground_truth_pair_share","beats_compound_baseline"]
print(prom[cols].round(4).to_string(index=False))
prom.to_csv("../results/DEMO_synthetic_promiscuity.csv", index=False)
print("\nover seeds:")
print(prom[["model","compound_baseline","within_variant_shuffled","pair_specificity"]]
      .agg(["mean","std"]).round(4).to_string())

2026-10-10 11:54:02,940 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1308 pairs, prevalence 0.312


2026-10-10 11:54:02,960 INFO    foldrescue.features: built features: (1308, 300) (float64)


2026-10-10 11:54:03,040 INFO    foldrescue.features: built features: (262, 300) (float64)


2026-10-10 11:54:03,091 INFO    foldrescue.attribution: promiscuity interrogation: PASSES: 56% of the model's lift over chance is destroyed by permuting compound identity within each variant, so most of what it knows is genuinely pair-specific rather than per-compound.


seed 0 verdict:
  PASSES: 56% of the model's lift over chance is destroyed by permuting compound identity within each variant, so most of what it knows is genuinely pair-specific rather than per-compound. 



...

2026-10-10 11:54:04,536 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1309 pairs, prevalence 0.312


2026-10-10 11:54:04,554 INFO    foldrescue.features: built features: (1309, 300) (float64)


2026-10-10 11:54:04,648 INFO    foldrescue.features: built features: (262, 300) (float64)


2026-10-10 11:54:04,712 INFO    foldrescue.attribution: promiscuity interrogation: MIXED: only 46% of the lift is pair-specific; the majority comes from per-compound effects. Useful for ranking compounds, weak for ranking variants within a compound.


 seed  model  compound_baseline  variant_baseline  within_variant_shuffled  prevalence  pair_specificity  ground_truth_pair_share  beats_compound_baseline
    0 0.5445             0.3735            0.3420                   0.4233      0.3282            0.5605                   0.3403                     True
    1 0.3441             0.2777            0.2969                   0.2675      0.2313            0.6788                   0.3403                     True
    2 0.3656             0.3428            0.3399                   0.3060      0.2835            0.7267                   0.3403                     True
    3 0.4624             0.4187            0.3512                   0.4187      0.3586            0.4216                   0.3403                     True
    4 0.4656             0.3592            0.2593                   0.3826      0.2863            0.4627                   0.3403                     True

over seeds:
       model  compound_baseline  within_variant_shuffled 

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.3))

ax = axes[0]
names = ["prevalence", "variant_baseline", "compound_baseline", "within_variant_shuffled", "model"]
pretty = ["no-skill\n(prevalence)", "variant\nbase rate", "compound\nbase rate",
          "model, pairing\ndestroyed", "model"]
means = [prom[n].mean() for n in names]
sds = [prom[n].std() for n in names]
colours = [INK_MUTED, INK_MUTED, SERIES[1], SERIES[3], SERIES[0]]
bars = ax.bar(range(len(names)), means, yerr=sds, capsize=3.5, width=0.64,
              color=colours, zorder=3)
label_bars(ax, bars, means, dy=max(sds)+0.008)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(pretty, fontsize=7.8)
ax.set_ylabel("PR-AUC (mean +/- SD over 5 seeds)")
ax.set_title("The model must beat the promiscuity baseline\n— and lose ground when the pairing is destroyed")
ax.set_ylim(0, max(means)*1.3)

ax = axes[1]
rec = prom["pair_specificity"].mean()
gt = prom["ground_truth_pair_share"].mean()
bars2 = ax.bar([0, 1], [rec, gt], yerr=[prom["pair_specificity"].std(), 0],
               capsize=3.5, width=0.5, color=[SERIES[0], INK_MUTED], zorder=3)
label_bars(ax, bars2, [rec, gt], dy=0.015)
ax.set_xticks([0, 1])
ax.set_xticklabels(["recovered by\nthe diagnostic", "generative\nground truth"], fontsize=8.2)
ax.set_ylabel("pair-specific share of signal")
ax.set_ylim(0, max(rec, gt)*1.45)
ax.set_title("Validating the diagnostic against known truth\n(only possible on synthetic data)")

fig.suptitle("Did the model learn pair-specific signal, or compound promiscuity?",
             y=1.05, fontsize=12, color=INK)
save(fig, "04_promiscuity_interrogation.png")
plt.show()
print("\nTable view:")
print(pd.DataFrame({"quantity": pretty, "PR-AUC": np.round(means,4), "SD": np.round(sds,4)}).to_string(index=False))

wrote results/figures/04_promiscuity_interrogation.png

Table view:
                 quantity  PR-AUC     SD
   no-skill\n(prevalence)  0.2976 0.0484
       variant\nbase rate  0.3179 0.0389
      compound\nbase rate  0.3544 0.0513
model, pairing\ndestroyed  0.3596 0.0697
                    model  0.4365 0.0818


### What to take from this

Read the printed verdict and tables above rather than a remembered summary. The
structural claims:

1. **Removing the ligand block is not a configuration, it is a lobotomy.**
   Without it the model cannot distinguish compounds at all, so every pair
   sharing a variant receives an identical score — it stops being a pair model.
   Its ablation number should be read as a floor, not a comparison.
2. **Retrain-without and permutation can disagree**, and where they do, the
   block was substitutable: the fitted model leaned on it, but equivalent
   information was reachable elsewhere when it was absent.
3. **Deltas smaller than the pooled seed SD are not real.** The
   `beyond_noise` column marks this per block, and some blocks do not clear it.
4. **The promiscuity interrogation is the load-bearing result in this
   notebook.** A model that fails it is a per-compound lookup table however good
   its PR-AUC and however pretty its SHAP plot.
5. **The recovered pair-signal share can be checked against generative truth
   here, and only here** — and in this run the check shows the diagnostic is
   **biased upward**: it recovers ~0.57 against a generative share of 0.34,
   consistently across all five seeds.

   That is worth stating rather than smoothing over. The two quantities are not
   the same estimand: `pair_specificity` measures the share of the *model's
   realised lift* that depends on the pairing, while `pair_signal_share` is the
   share of *generative logit variance* carried by the interaction term. They
   need not coincide, and the gap here has a plausible cause — the
   within-variant shuffle also disrupts the stability × ligand interaction
   features, so some stability-driven signal is charged to pair specificity.

   The practical reading: treat `pair_specificity` as an **upper bound** on
   genuine pair-specific signal, not a point estimate. A model that fails the
   test on this metric has certainly failed; one that passes narrowly may not
   have. On a real benchmark the right-hand panel is unavailable entirely —
   which is the argument for keeping a synthetic cohort in the repository
   permanently, as a test fixture for the diagnostics themselves.
